In [6]:
import pandas as pd
import statsmodels.formula.api as smf
from pkg import nakagawa_r2
import os
os.chdir("/Users/caropark/FAO_ag_check_code")

if __name__ == "__main__":
    
    ### Read in data
    df = pd.read_csv("./data/yield_comparison_df.csv")
    
    ### Transform X covariates for modelling
    coeffs = ["avg_gdp","flag_sum","total_harvarea","cropland_fraction","avg_sm","avg_tmax", "whichlag"]
    df[coeffs].drop(columns="whichlag").agg(['min', 'max'])
    scaled = df[coeffs].drop(columns="whichlag").apply(lambda x: (x - x.mean()) / x.std()).add_suffix("_z")
    df = df.drop(columns=df[coeffs].drop(columns="whichlag")).join(scaled)

    ### Main model
    mod_yields_r2 = smf.mixedlm("r2 ~ avg_gdp_z + flag_sum_z + C(whichlag, Treatment(reference='yield_log_dt')) + total_harvarea_z + cropland_fraction_z ", # + avg_sm_z + avg_tmax_z",
                        df, groups="cropname" )
    print(mod_yields_r2.fit().summary())
    nakagawa_r2(mod_yields_r2.fit())

    ### Gnerate model table for Latex
    res = mod_yields_r2.fit()
    params = res.params
    conf = res.conf_int()
    summary_df = (
        pd.DataFrame({
            "Coef.": params,
            "Std.Err.": res.bse,
            "t": res.tvalues,
            "P>|t|": res.pvalues,
            "CI Lower": conf[0],
            "CI Upper": conf[1]
        }))

    print(summary_df.to_latex(float_format="%.3f"))



                                  Mixed Linear Model Regression Results
Model:                              MixedLM                  Dependent Variable:                  r2      
No. Observations:                   1744                     Method:                              REML    
No. Groups:                         19                       Scale:                               0.0189  
Min. group size:                    47                       Log-Likelihood:                      945.5487
Max. group size:                    150                      Converged:                           Yes     
Mean group size:                    91.8                                                                  
----------------------------------------------------------------------------------------------------------
                                                               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------

/opt/homebrew/Caskroom/miniforge/base/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/Caskroom/miniforge/base/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/homebrew/Caskroom/miniforge/base/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/var/folders/k4/2gcvspsj7d7c9hbxlvzv_zrh0000gn/T/ipykernel_72963/640181043.py:38: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended in